In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="01-ml-design-prep",
    notebook_name="02_interview_strategy.ipynb"
)

# ML System Design Interview Strategy & Practice

## The 12-Year-Old Version

You've learned the rules of chess. Now it's time to practice actually *playing* the game.

This notebook is your sparring partner. It covers:
- How to manage 45 minutes without running out of time
- How to communicate at a staff level (not just what to say, but HOW)
- Mini walkthroughs of 3 practice problems
- 50 key vocabulary terms that signal expertise

Let's get you interview-ready\!

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

# ============================================================
# Time Management: 45-minute Interview Breakdown
# ============================================================

phases = [
    ("Clarify Requirements",    5,  "#FFE0B2"),
    ("Frame ML Task",           5,  "#B3E5FC"),
    ("Data + Features",         8,  "#C8E6C9"),
    ("Model Development",      12,  "#F8BBD0"),
    ("Evaluation",              5,  "#D1C4E9"),
    ("Serving Architecture",    7,  "#FFF9C4"),
    ("Follow-ups / Deep Dives", 3,  "#FFCCBC"),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gantt-style timeline
cumulative = 0
for phase, mins, color in phases:
    ax1.barh(0, mins, left=cumulative, color=color, edgecolor='#333', linewidth=1.5, height=0.5)
    ax1.text(cumulative + mins/2, 0, f'{phase}\n({mins}m)',
             ha='center', va='center', fontsize=7.5, fontweight='bold', wrap=True)
    cumulative += mins

ax1.set_xlim(0, 47)
ax1.set_yticks([])
ax1.set_xlabel('Minutes', fontsize=11)
ax1.set_title('45-Minute Interview Timeline', fontsize=13, fontweight='bold')
ax1.axvline(45, color='red', linestyle='--', linewidth=2, label='Time limit')
ax1.legend()

# Pie chart
labels = [p[0].split()[0] for p in phases]
mins   = [p[1] for p in phases]
colors = [p[2] for p in phases]
ax2.pie(mins, labels=labels, colors=colors, autopct='%1.0f%%',
        startangle=90, pctdistance=0.75, textprops={'fontsize': 9})
ax2.set_title('Time Allocation by Phase', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("Time management rules:")
print("  - Never spend >10 min on requirements (interviewer will be bored)")
print("  - Model discussion = longest section; go deep here")
print("  - If running out of time, say: 'I'd next cover serving -- want me to go there?'")

## Communication Templates

What you say matters as much as what you know. Here are templates that signal staff-level thinking.

### How to Start (First 30 Seconds)

In [ ]:
templates = {
    "Opening": [
        "WEAK: 'I'll design a recommendation system.'",
        "STRONG: 'Before diving in, let me clarify a few things to make sure I'm solving the right problem...'",
        "WHY: Shows you don't make assumptions. Interviewers love this.",
    ],
    "Proposing your approach": [
        "WEAK: 'I'll use a neural network.'",
        "STRONG: 'I see two main approaches here. Option A is X which trades off Y for Z. Option B does the opposite. Given our constraints, I'd lean toward A because...'",
        "WHY: Demonstrates awareness of the design space, not just one answer.",
    ],
    "Discussing trade-offs": [
        "WEAK: 'FAISS is good for ANN search.'",
        "STRONG: 'We have a few options here: exact search is O(N*D) which won't scale to 200B images; tree-based methods degrade in high dimensions; so I'd use FAISS IVFFlat which gets us O(D*log N) at the cost of ~5% recall loss -- acceptable for this use case.'",
        "WHY: Shows depth. Interviewers want to see you understand WHY, not just WHAT.",
    ],
    "Handling unknown": [
        "WEAK: (silence or making something up)",
        "STRONG: 'I haven't worked with X directly, but from first principles I'd approach it as... and I'd validate by...'",
        "WHY: Honesty + reasoning ability impresses more than bluffing.",
    ],
    "Time transitions": [
        "WEAK: (keep talking about data forever)",
        "STRONG: 'I think I've covered the key data decisions. Should I move on to model architecture, or would you like me to go deeper on any of this?'",
        "WHY: Shows self-awareness and respects the interviewer's time.",
    ],
}

for topic, lines in templates.items():
    print(f"{'='*60}")
    print(f"  {topic}")
    print(f"{'='*60}")
    for line in lines:
        print(f"  {line}")
    print()

## Mini-Walkthroughs: 3 Practice Problems

Apply the 7-step framework quickly to 3 problems. Each walkthrough is condensed to show the skeleton -- flesh it out when practicing.

In [ ]:
problems = [
    {
        "name": "Spam Detection (Email)",
        "steps": {
            "Clarify":  "Email spam vs legit? Any language? False positive cost (legit -> spam) vs false negative (spam -> inbox)?",
            "Frame":    "Binary classification. Input=email (subject+body+sender). Output=P(spam). ML obj=maximize precision while keeping recall>95%.",
            "Data":     "Email corpus with labels. Features: TF-IDF of body, sender reputation score, domain age, link count, urgency words.",
            "Model":    "Baseline: Naive Bayes (fast, works well for text). Complex: Fine-tuned BERT. Ensemble for production.",
            "Eval":     "Offline: PR-AUC (imbalanced\!), precision@recall=0.99. Online: spam complaints, false positive rate.",
            "Serving":  "Online prediction at receive time. <100ms required. Cache sender reputation scores.",
            "Monitor":  "Spammers adapt. Retrain weekly. Monitor for new spam patterns (topic drift).",
        }
    },
    {
        "name": "ETA Prediction (Ride-sharing)",
        "steps": {
            "Clarify":  "Pickup ETA or trip ETA? City or global? Accuracy target? How fresh is traffic data?",
            "Frame":    "Regression. Input=(origin, destination, time, traffic). Output=seconds. ML obj=minimize MAE.",
            "Data":     "Historical trips (lat/lon, time, duration). Real-time traffic API. Weather API. Features: distance, time-of-day, day-of-week, historical speed on route segments.",
            "Model":    "Baseline: distance/avg_speed. Complex: GBDT on route features. Deep: Graph NN on road network.",
            "Eval":     "Offline: MAE, RMSE, MAPE. Online: % of predictions within ±2 min, driver/rider satisfaction ratings.",
            "Serving":  "Online prediction, <200ms. Pre-compute popular route embeddings. Batch-update route stats hourly.",
            "Monitor":  "Degrade during events (concerts, sports). Alert if MAE spikes >20%. Retrain daily with fresh trip data.",
        }
    },
    {
        "name": "People You May Know (Social Network)",
        "steps": {
            "Clarify":  "Bidirectional friendships? What signals? Minimize 'creepy' recommendations? How many suggestions shown?",
            "Frame":    "Ranking/binary classification. Input=user. Output=ranked list of non-friends by P(friend request accepted).",
            "Data":     "Friendship graph. Interaction logs (messages, likes, profile views). Features: common friends, Jaccard similarity, mutual events, school/work overlap.",
            "Model":    "Baseline: rank by #common friends. Complex: GNN (GraphSAGE) over social graph. Feature engineering critical.",
            "Eval":     "Offline: mAP (did we rank eventual friends highly?). Online: friend requests sent+accepted per day.",
            "Serving":  "Batch pre-compute nightly (users don't need real-time). Cache top-50 candidates per user.",
            "Monitor":  "Privacy: filter out blocked users. Track acceptance rate -- if drops, model may be getting 'creepy'.",
        }
    }
]

for prob in problems:
    print(f"\n{'='*65}")
    print(f"  PROBLEM: {prob['name']}")
    print(f"{'='*65}")
    for step, answer in prob['steps'].items():
        print(f"  [{step:8s}] {answer}")

## Role-Specific Focus Areas

Different roles emphasize different steps. Know your audience\!

In [ ]:
roles = pd.DataFrame([
    ["Data Scientist",        "High",   "High",  "Medium", "Medium", "Low",  "Low"],
    ["Applied ML Engineer",   "Medium", "High",  "High",   "High",   "High", "Medium"],
    ["ML Infrastructure",     "Low",    "Medium","Low",    "Medium", "High", "High"],
    ["Research Scientist",    "Low",    "High",  "Low",    "High",   "Medium","Low"],
    ["Staff ML Engineer",     "High",   "High",  "High",   "High",   "High", "High"],
], columns=["Role", "Clarify", "Model", "Data/Features", "Training", "Serving", "Monitoring"])

print("=== Role-Specific Focus Areas ===")
print(roles.to_string(index=False))
print()
print("For Staff Engineer: you need to go DEEP on ALL sections.")
print("That's what separates staff from senior.")

## 50 Key ML Vocabulary Terms

Drop these in interviews to signal depth. Know what each one means and when to use it.

In [ ]:
vocab = pd.DataFrame([
    # Architecture
    ["Two-Tower Model",       "Architecture",  "Dual encoder for retrieval: user tower + item tower, dot product similarity"],
    ["Shared + Task Heads",   "Architecture",  "Multi-task learning: shared layers learn common representations, heads specialize"],
    ["Early Fusion",          "Architecture",  "Combine multi-modal features before the main model (vs. late fusion after)"],
    ["Feature Store",         "Infrastructure","Centralized repo for pre-computed features; online (low latency) + offline"],
    ["Two-Stage Pipeline",    "Architecture",  "Candidate generation (fast/rough) + ranking (slow/precise)"],
    # Training
    ["Contrastive Loss",      "Training",      "NT-Xent / InfoNCE: push positives together, negatives apart in embedding space"],
    ["Focal Loss",            "Training",      "Down-weight easy examples, focus on hard ones -- great for class imbalance"],
    ["WALS",                  "Training",      "Weighted Alternating Least Squares for matrix factorization; faster than SGD"],
    ["Fine-tuning",           "Training",      "Start from pretrained weights, continue training on domain-specific data"],
    ["Negative Sampling",     "Training",      "Sample negatives explicitly during training for contrastive/CF models"],
    # Data
    ["Natural Labels",        "Data",          "Labels derived automatically from user actions (clicks, purchases, time-spent)"],
    ["Feature Cross",         "Data",          "Explicit combination of two features: e.g., age × device_type"],
    ["Embedding Layer",       "Data",          "Learns dense vector representations for high-cardinality categoricals"],
    ["Data Augmentation",     "Data",          "Artificially expand training data via transformations (flip, crop, rotate)"],
    ["Hard Negative Mining",  "Data",          "Explicitly add difficult negatives to training data to improve model robustness"],
    # Evaluation
    ["nDCG",                  "Evaluation",    "Normalized DCG: ranking quality metric for graded relevance, normalized to [0,1]"],
    ["PR-AUC",                "Evaluation",    "Area under precision-recall curve; better than ROC-AUC for imbalanced classes"],
    ["mAP",                   "Evaluation",    "Mean Average Precision; ranking quality for binary relevance across multiple queries"],
    ["Calibration",           "Evaluation",    "Predicted probabilities match empirical frequencies; critical for ad bidding"],
    ["A/B Testing",           "Evaluation",    "Randomized experiment: route traffic to old vs new model, measure real metrics"],
    # Search/Retrieval
    ["ANN Search",            "Retrieval",     "Approximate Nearest Neighbor: trade tiny recall loss for huge speed gain"],
    ["FAISS",                 "Retrieval",     "Facebook AI Similarity Search; supports IVF, HNSW, PQ indexing at scale"],
    ["IVF Index",             "Retrieval",     "Inverted File Index: cluster vectors, search only nearest clusters"],
    ["Product Quantization",  "Retrieval",     "Compress embeddings by quantizing sub-vectors; 32x memory reduction possible"],
    ["HNSW",                  "Retrieval",     "Hierarchical Navigable Small World: graph-based ANN, excellent recall/speed"],
    # Production
    ["Shadow Deployment",     "Serving",       "Run new model alongside old, only serve old results but compare both"],
    ["Canary Release",        "Serving",       "Gradually roll out new model to increasing % of traffic"],
    ["Online Prediction",     "Serving",       "Generate predictions at request time (vs. batch pre-compute)"],
    ["Model Distillation",    "Serving",       "Train small student model to mimic large teacher model; compression technique"],
    ["Quantization",          "Serving",       "Reduce model weight precision (32-bit → 8-bit) to shrink size and speed up inference"],
    # Monitoring
    ["Distribution Shift",    "Monitoring",    "Training data distribution ≠ production data; major cause of model degradation"],
    ["Concept Drift",         "Monitoring",    "The underlying relationship between features and labels changes over time"],
    ["Positional Bias",       "Monitoring",    "Users click items at top regardless of relevance; must correct in training"],
    ["IPW Correction",        "Monitoring",    "Inverse Propensity Weighting: up-weight examples shown in low positions"],
    ["Retraining Frequency",  "Monitoring",    "How often to retrain; depends on drift speed. Daily/weekly common in production"],
    # System Design
    ["Cold Start",            "System",        "New user/item with no interaction history; need fallback strategies"],
    ["Long-Tail Problem",     "System",        "Most items rarely seen; model underperforms on them without special handling"],
    ["Exploration vs Exploit","System",        "Balance showing proven content (exploit) vs. testing new content (explore)"],
    ["Thundering Herd",       "System",        "Viral event causes massive traffic spike; need caching/rate limiting"],
    ["Backfill",              "System",        "Process historical data to populate a feature store or retrain a model"],
    # Models
    ["DeepFM",                "Model",         "Deep Factorization Machine: FM + DNN for CTR prediction; handles feature interactions"],
    ["DCN",                   "Model",         "Deep & Cross Network: explicit feature crossing via cross layers"],
    ["GraphSAGE",             "Model",         "Inductive GNN: aggregate neighborhood features; works on unseen nodes"],
    ["Faster R-CNN",          "Model",         "Two-stage object detector: RPN proposes regions, classifier refines + localizes"],
    ["SimCLR",                "Model",         "Self-supervised contrastive learning using data augmentation as positives"],
    ["Two-Tower (MF mode)",   "Model",         "Matrix factorization as two-tower: user/item ID embeddings, dot product"],
    ["MMR",                   "Ranking",       "Maximal Marginal Relevance: balance relevance + diversity in re-ranking"],
    ["LambdaRank",            "Ranking",       "Pairwise LTR: gradient computed from nDCG differences between swapped pairs"],
    ["BPR",                   "Ranking",       "Bayesian Personalized Ranking: optimize relative ordering (pos > neg)"],
    ["Weighted BCE",          "Training",      "Binary cross-entropy with class weights; addresses class imbalance"],
    ["Gradient Blending",     "Training",      "Balance gradient magnitudes across modalities to prevent one dominating"],
], columns=["Term", "Category", "Definition"])

# Print grouped by category
for cat in vocab['Category'].unique():
    group = vocab[vocab['Category'] == cat]
    print(f"\n--- {cat} ---")
    for _, row in group.iterrows():
        print(f"  {row['Term']:30s} {row['Definition']}")

## Self-Assessment Checklist

Use this before your interview to make sure you're ready:

In [ ]:
checklist = [
    ("Requirements", [
        "Can I ask 6 types of clarifying questions without prompting?",
        "Do I know how to connect business objectives to ML objectives?",
        "Can I summarize requirements clearly before moving on?",
    ]),
    ("ML Framing", [
        "Can I frame any problem as classification/regression/ranking?",
        "Do I know the input/output for all 11 case studies?",
        "Can I explain WHY I chose a particular ML category?",
    ]),
    ("Data & Features", [
        "Can I describe the feature engineering for 5+ case studies?",
        "Do I know how to handle missing values, scaling, and encoding?",
        "Can I explain embedding layers vs. one-hot encoding?",
    ]),
    ("Models", [
        "Can I explain 4+ model architectures (two-tower, multi-task, GBDT, DeepFM)?",
        "Do I know which loss function to use for each task type?",
        "Can I explain contrastive learning from scratch?",
    ]),
    ("Evaluation", [
        "Can I compute nDCG, mAP, PR-AUC from scratch?",
        "Do I know offline vs online metrics for all 11 case studies?",
        "Can I explain when to use each metric?",
    ]),
    ("Serving", [
        "Can I draw the serving architecture for 5+ case studies?",
        "Do I know how A/B testing, shadow deployment, and canary releases work?",
        "Can I explain the candidate generation → ranking → re-ranking funnel?",
    ]),
    ("Staff-Level Signals", [
        "Do I discuss trade-offs for EVERY decision I make?",
        "Can I estimate scale (QPS, storage, GPU requirements)?",
        "Do I proactively bring up cold start, bias, privacy, fairness?",
    ]),
]

total = sum(len(items) for _, items in checklist)
print(f"Self-Assessment Checklist ({total} items)")
print("Rate yourself: [3] Confident  [2] Shaky  [1] Need review  [0] Don't know\n")

for section, items in checklist:
    print(f"=== {section} ===")
    for item in items:
        print(f"  [ ] {item}")
    print()

print(f"Target score: {total*3} points")
print("If any section has a 0: review that case study notebook before the interview\!")

## Final Review: All 11 Case Studies

Quick reference cross-linking all case studies in this repo:

In [ ]:
case_studies = pd.DataFrame([
    ["01", "ML Design Framework",   "Framework",        "N/A",               "This notebook\!"],
    ["02", "Visual Search",          "Representation Learning", "nDCG",      "Contrastive, SimCLR, FAISS, ANN"],
    ["03", "Google Street View",     "Object Detection", "mAP, IoU",         "Faster R-CNN, NMS, hard negatives"],
    ["04", "YouTube Video Search",   "Multimodal Ranking","nDCG, MRR",       "Two-tower, text+video embeddings"],
    ["05", "Harmful Content",        "Multi-task Classification","PR-AUC,ROC-AUC","Early fusion, DistilBERT, CLIP"],
    ["06", "Video Recommendation",   "Hybrid Filtering", "Precision@k, diversity","Matrix factorization, two-tower, WALS"],
    ["07", "Event Recommendation",   "Pointwise LTR",    "mAP",              "XGBoost→NN, 5 feature categories, continual learning"],
    ["08", "Ad Click Prediction",    "Binary Classification","Calibration, AUC","DeepFM, DCN, DIN, feature crosses"],
    ["09", "Similar Listings",       "Representation Learning","Avg rank of booked","Listing2Vec, session embeddings"],
    ["10", "News Feed",              "Multi-task Ranking","ROC-AUC per task", "Multi-task DNN, dwell-time, affinity features"],
    ["11", "People You May Know",    "Graph-based Ranking","mAP",            "GCN, Node2Vec, Adamic-Adar, privacy filters"],
], columns=["Module", "Problem", "ML Category", "Key Metric", "Key Concepts"])

print("=== All 11 ML Design Case Studies ===")
print(case_studies.to_string(index=False))
print()
print("Study order recommendation:")
print("  1. Read this notebook (framework)")
print("  2. Visual Search (representation learning foundation)")
print("  3. Event Recommendation (full LTR example)")
print("  4. Video Recommendation (CF + neural ranking)")
print("  5. Then any order based on your interview focus")

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)